In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
from tqdm.notebook import tqdm
import gc
import json

plt.style.use("ggplot")

In [2]:
# Connect to database
conn = sqlite3.connect("mimic.db")

# Confirm connection
pd.read_sql("PRAGMA database_list;", conn)

,seq,name,file
0,0,main,/blue/cap5771/shesadree.p/CAP5771_lili1501/mim...


In [3]:
import pandas as pd

pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn)

,name
0,inputevents
1,outputevents
2,chartevents
3,patients
4,d_icd_diagnoses
5,d_icd_procedures
6,d_labitems
7,d_items
8,admissions
9,diagnoses_icd


In [5]:
schema = """
CREATE TABLE patients (
    subject_id INTEGER PRIMARY KEY,
    gender TEXT,
    anchor_age INTEGER,
    anchor_year INTEGER,
    anchor_year_group TEXT,
    dod TEXT
);

CREATE TABLE admissions (
    hadm_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    admittime TEXT,
    dischtime TEXT,
    deathtime TEXT,
    admission_type TEXT,
    admit_provider_id TEXT,
    admission_location TEXT,
    discharge_location TEXT,
    insurance TEXT,
    language TEXT,
    marital_status TEXT,
    race TEXT,
    edregtime TEXT,
    edouttime TEXT,
    hospital_expire_flag INTEGER,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id)
);

CREATE TABLE diagnoses_icd (
    subject_id INTEGER,
    hadm_id INTEGER,
    seq_num INTEGER,
    icd_code TEXT,
    icd_version INTEGER,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
);



CREATE TABLE labevents (
    labevent_id INTEGER NOT NULL,
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    specimen_id INTEGER NOT NULL,
    itemid INTEGER NOT NULL,
    order_provider_id TEXT,
    charttime TEXT,
    storetime TEXT,
    value TEXT,
    valuenum REAL,
    valueuom TEXT,
    ref_range_lower REAL,
    ref_range_upper REAL,
    flag TEXT,
    priority TEXT,
    comments TEXT,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
);
"""
conn.executescript(schema)


In [6]:
schema = """

CREATE TABLE d_icd_diagnoses (
    icd_code TEXT NOT NULL,
    icd_version INTEGER NOT NULL,
    long_title TEXT,
    PRIMARY KEY (icd_code, icd_version)
);


CREATE TABLE d_icd_procedures (
    icd_code TEXT NOT NULL,
    icd_version INTEGER NOT NULL,
    long_title TEXT,
    PRIMARY KEY (icd_code, icd_version)
);


CREATE TABLE d_labitems (
    itemid INTEGER PRIMARY KEY,
    label TEXT,
    fluid TEXT,
    category TEXT
);


CREATE TABLE d_items (
    itemid INTEGER PRIMARY KEY,
    label TEXT,
    abbreviation TEXT,
    linksto TEXT,
    category TEXT,
    unitname TEXT,
    param_type TEXT,
    lownormalvalue REAL,
    highnormalvalue REAL
);

"""
conn.executescript(schema)


In [7]:
schema = """

CREATE TABLE icustays (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER NOT NULL,
    first_careunit TEXT,
    last_careunit TEXT,
    intime TEXT,
    outtime TEXT,
    los REAL,
    PRIMARY KEY (stay_id)
);


CREATE TABLE inputevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    starttime TEXT,
    endtime TEXT,
    storetime TEXT,
    itemid INTEGER,
    amount REAL,
    amountuom TEXT,
    rate REAL,
    rateuom TEXT,
    orderid INTEGER,
    linkorderid INTEGER,
    ordercategoryname TEXT,
    secondaryordercategoryname TEXT,
    ordercomponenttypedescription TEXT,
    ordercategorydescription TEXT,
    patientweight REAL,
    totalamount REAL,
    totalamountuom TEXT,
    isopenbag INTEGER,
    continueinnextdept INTEGER,
    statusdescription TEXT,
    originalamount REAL,
    originalrate REAL
);


CREATE TABLE outputevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    charttime TEXT,
    storetime TEXT,
    itemid INTEGER,
    value REAL,
    valueuom TEXT
);



CREATE TABLE chartevents (
    subject_id INTEGER NOT NULL,
    hadm_id INTEGER,
    stay_id INTEGER,
    caregiver_id INTEGER,
    charttime TEXT,
    storetime TEXT,
    itemid INTEGER,
    value TEXT,
    valuenum REAL,
    valueuom TEXT,
    warning INTEGER
);


"""
conn.executescript(schema)


In [8]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';",conn)

,name
0,patients
1,admissions
2,diagnoses_icd
3,labevents
4,d_icd_diagnoses
5,d_icd_procedures
6,d_labitems
7,d_items
8,icustays
9,inputevents


In [4]:
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "mimic dataset"

In [82]:
def load_csv_gz(path, table, subject_set=None, item_ids=None, chunksize=None):

    print(f"Loading {path.name} → {table}")

    read_cols = None  # optional: reduce memory by selecting columns

    if chunksize:
        for chunk in tqdm(
            pd.read_csv(path, compression="gzip", chunksize=chunksize),
            desc=table
        ):

            # Filter subject_id
            if subject_set is not None and "subject_id" in chunk.columns:
                chunk = chunk[chunk["subject_id"].isin(subject_set)]

            # Filter itemid (labs or chart events)
            if item_ids is not None and "itemid" in chunk.columns:
                chunk = chunk[chunk["itemid"].isin(item_ids)]

            if not chunk.empty:
                chunk.to_sql(table, conn, if_exists="append", index=False)

            del chunk
            gc.collect()

    else:
        df = pd.read_csv(path, compression="gzip")

        if subject_set is not None and "subject_id" in df.columns:
            df = df[df["subject_id"].isin(subject_set)]

        if item_ids is not None and "itemid" in df.columns:
            df = df[df["itemid"].isin(item_ids)]

        if not df.empty:
            df.to_sql(table, conn, if_exists="replace", index=False)

In [25]:
load_csv_gz(DATA_DIR / "hosp/patients.csv.gz", "patients")

Loading patients.csv.gz → patients


In [26]:
sampled_subjects = pd.read_sql("""
SELECT subject_id
FROM patients
ORDER BY RANDOM()
LIMIT (
    SELECT CAST(COUNT(DISTINCT subject_id) * 0.5 AS INT)
    FROM patients
);
""", conn)

print(f"Sampled {len(sampled_subjects)} subjects.")
sampled_subjects.head()

Sampled 182313 subjects.


,subject_id
0,12047698
1,10449138
2,14224902
3,10380387
4,15291237


In [27]:
load_csv_gz(DATA_DIR / "hosp/d_icd_diagnoses.csv.gz", "d_icd_diagnoses")
load_csv_gz(DATA_DIR / "hosp/d_icd_procedures.csv.gz", "d_icd_procedures")
load_csv_gz(DATA_DIR / "hosp/d_labitems.csv.gz", "d_labitems")


Loading d_icd_diagnoses.csv.gz → d_icd_diagnoses
Loading d_icd_procedures.csv.gz → d_icd_procedures
Loading d_labitems.csv.gz → d_labitems


In [28]:
#icu tables
load_csv_gz(DATA_DIR / "icu/d_items.csv.gz", "d_items")

Loading d_items.csv.gz → d_items


##### charevents

In [26]:
hr_ids = pd.read_sql("""
SELECT itemid
FROM d_items
WHERE LOWER(label) = 'heart rate'
""", conn)["itemid"].tolist()

print(hr_ids)

[220045]


In [27]:
map_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) IN ('arterial blood pressure mean','non invasive blood pressure mean')
""", conn)["itemid"].tolist()

print(map_ids)

[220052, 220181]


In [28]:
rr_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) IN (
    'respiratory rate',
    'respiratory rate (spontaneous)',
    'respiratory rate (total)'
)
""", conn)["itemid"].tolist()

print(rr_ids)

[220210, 224689, 224690]


In [29]:
sbp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN ('arterial blood pressure systolic','non invasive blood pressure systolic')
""", conn)["itemid"].tolist()

print(sbp_ids)

dbp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN ('arterial blood pressure diastolic','non invasive blood pressure diastolic')
""", conn)["itemid"].tolist()

print(dbp_ids)

[220050, 220179]
[220051, 220180]


In [30]:
# oxygen saturation itemids
spo2_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) = 'o2 saturation pulseoxymetry'
""", conn)["itemid"].tolist()

print(spo2_ids)

[220277]


In [31]:
fio2_ids = pd.read_sql("""
SELECT itemid
FROM  d_items 
WHERE LOWER(label) = 'inspired o2 fraction'
""", conn)["itemid"].tolist()

print(fio2_ids)

[223835]


In [32]:
pao2_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) = 'arterial o2 pressure'
""", conn)["itemid"].tolist()

print(pao2_ids)

[220224]


In [33]:

temp_ids = pd.read_sql("""
SELECT itemid
FROM d_items 
WHERE LOWER(label) IN (
    'temperature fahrenheit',
    'temperature celsius'
)
""", conn)["itemid"].tolist()

print(temp_ids)

[223761, 223762]


##### labevents

In [34]:
creatinine_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN ('creatinine')
""", conn)["itemid"].tolist()

print(creatinine_ids)

[50912, 52546]


In [35]:
lactate_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) = 'lactate'
""", conn)["itemid"].tolist()

print(lactate_ids)

[50813, 52442, 53154]


In [36]:
bilirubin_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) = 'bilirubin, total'
""", conn)["itemid"].tolist()

print(bilirubin_ids)

[50885, 53089]


In [37]:
wbc_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) IN ('wbc', 'wbc count','white blood cells')
""", conn)["itemid"].tolist()

print(wbc_ids)

[51300, 51301, 51516, 51755, 51756, 52407]


In [38]:
platelet_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems 
WHERE LOWER(label) = 'platelet count'
""", conn)["itemid"].tolist()

print(platelet_ids)

[51265, 53189]


In [96]:
hemog_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN( 'hemoglobin', 'absolute hemoglobin','hemoglobin, calculated')
""", conn)["itemid"].tolist()

print(hemog_ids)

sodium_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems
WHERE LOWER(label) IN ('sodium', 'sodium, whole blood')
""", conn)["itemid"].tolist()

print(sodium_ids)

potassium_ids = pd.read_sql("""
SELECT itemid
FROM  d_labitems 
WHERE LOWER(label) IN ('potassium', 'potassium, whole blood')
""", conn)["itemid"].tolist()

print(potassium_ids)

bun_ids = pd.read_sql("""
SELECT itemid
FROM d_labitems
WHERE LOWER(label) IN ('urea nitrogen', 'bun')
""", conn)["itemid"].tolist()

print(bun_ids)

[50811, 50855, 51222, 51640, 51645]
[50824, 50983, 52455, 52623]
[50822, 50833, 50971, 52452, 52610]
[51006, 51842, 52647]


In [59]:
chartevent_ids = {
    "hr": hr_ids,
    "rr": rr_ids,
    "map": map_ids,
    "sbp": sbp_ids,
    "dbp": dbp_ids,
    "fio2": fio2_ids,
    "pao2": pao2_ids,
    "spo2": spo2_ids,
    "temp": temp_ids
}

rows = []

for feature, ids in chartevent_ids.items():
    for itemid in ids:
        rows.append({"feature": feature, "itemid": itemid})

df = pd.DataFrame(rows)

df.to_csv("Itemids/chartevents_ids.csv", index=False)

In [97]:
labevent_ids = {
    "creatinine": creatinine_ids,
    "lactate": lactate_ids,
    "bilirubin": bilirubin_ids,
    "wbc": wbc_ids,
    "platelet": platelet_ids,
    "hemoglobin": hemog_ids,
    "sodium": sodium_ids,
    "potassium": potassium_ids,
    "bun": bun_ids
}

rows = []

for feature, ids in labevent_ids.items():
    for itemid in ids:
        rows.append({"feature": feature, "itemid": itemid})

df = pd.DataFrame(rows)

df.to_csv("Itemids/labevents_ids.csv", index=False)

In [98]:
lab_id_df = pd.read_csv("Itemids/labevents_ids.csv")

labevent_ids = lab_id_df.groupby("feature")["itemid"].apply(list).to_dict()
labevent_ids

chart_id_df = pd.read_csv("Itemids/chartevents_ids.csv")
chartevent_ids = chart_id_df.groupby("feature")["itemid"].apply(list).to_dict()
chartevent_ids
labevent_ids

{'bilirubin': [50885, 53089],
 'bun': [51006, 51842, 52647],
 'creatinine': [50912, 52546],
 'hemoglobin': [50811, 50855, 51222, 51640, 51645],
 'lactate': [50813, 52442, 53154],
 'platelet': [51265, 53189],
 'potassium': [50822, 50833, 50971, 52452, 52610],
 'sodium': [50824, 50983, 52455, 52623],
 'wbc': [51300, 51301, 51516, 51755, 51756, 52407]}

In [80]:
lab_item_ids = lab_id_df["itemid"].tolist()
char_item_ids = chart_id_df["itemid"].tolist()
char_item_ids


[220045,
 220210,
 224689,
 224690,
 220052,
 220181,
 220050,
 220179,
 220051,
 220180,
 223835,
 220224,
 220277,
 223761,
 223762]

In [47]:
subject_set = set(sampled_subjects["subject_id"])

In [48]:
load_csv_gz(DATA_DIR / "hosp/admissions.csv.gz", "admissions", subject_set=subject_set)
load_csv_gz(DATA_DIR / "hosp/diagnoses_icd.csv.gz", "diagnoses_icd", subject_set=subject_set)
load_csv_gz(DATA_DIR / "icu/icustays.csv.gz", "icustays", subject_set=subject_set)

Loading admissions.csv.gz → admissions
Loading diagnoses_icd.csv.gz → diagnoses_icd
Loading icustays.csv.gz → icustays


In [49]:
pd.read_sql("PRAGMA foreign_keys;", conn)

,foreign_keys
0,1


In [100]:
conn.execute("PRAGMA foreign_keys = OFF;")

In [ ]:
load_csv_gz(DATA_DIR / "hosp/labevents.csv.gz","labevents", subject_set=subject_set,item_ids=lab_item_ids,chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/inputevents.csv.gz", "inputevents", subject_set=subject_set, chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/outputevents.csv.gz", "outputevents", subject_set=subject_set, chunksize=50_000)
load_csv_gz(DATA_DIR / "icu/chartevents.csv.gz", "chartevents", subject_set=subject_set, item_ids=char_item_ids,chunksize=50_000)

Loading inputevents.csv.gz → inputevents


inputevents: 0it [00:00, ?it/s]

Loading outputevents.csv.gz → outputevents


outputevents: 0it [00:00, ?it/s]

Loading chartevents.csv.gz → chartevents


chartevents: 0it [00:00, ?it/s]

In [85]:
subject_set = set(
    pd.read_sql("SELECT DISTINCT subject_id FROM admissions", conn)["subject_id"]
)
# subject_set

In [101]:
bun_ids = [51006, 51842, 52647]

load_csv_gz(
    DATA_DIR / "hosp/labevents.csv.gz",
    "labevents",
    subject_set=subject_set,
    item_ids=bun_ids,
    chunksize=50_000
)

Loading labevents.csv.gz → labevents


labevents: 0it [00:00, ?it/s]

In [ ]:
load_csv_gz(
    DATA_DIR / "icu/chartevents.csv.gz",
    "chartevents",
    subject_set=subject_set,
    item_ids=[220277],   # only SpO2
    chunksize=50_000
)

Loading labevents.csv.gz → labevents


labevents: 0it [00:00, ?it/s]

Loading chartevents.csv.gz → chartevents


chartevents: 0it [00:00, ?it/s]

In [93]:
bun_df = pd.read_sql("""
SELECT itemid, label
FROM d_labitems
WHERE LOWER(label) IN ('urea nitrogen', 'bun')
""", conn)

print(bun_df)

   itemid          label
0   51006  Urea Nitrogen
1   51842            Bun
2   52647  Urea Nitrogen


In [105]:
pd.read_sql("""
SELECT COUNT(*)
FROM chartevents

""", conn)
# WHERE itemid in (51006, 51842, 52647)
# WHERE itemid = 220277

,COUNT(*)
0,28550391


In [6]:
conn.execute("PRAGMA foreign_keys = ON;")

In [7]:
df = pd.read_sql("SELECT * FROM admissions", conn)
print(len(df))

273865


#### Creating updated intermiediate view 

In [106]:
conn.executescript("""
DROP VIEW IF EXISTS chartevents_72h;

CREATE VIEW chartevents_72h AS
SELECT c.*
FROM chartevents c
JOIN icustays s USING (stay_id)
WHERE datetime(c.charttime) >= datetime(s.intime)
  AND datetime(c.charttime) <= datetime(s.intime, '+72 hours');
""")


In [108]:
conn.executescript("""
DROP VIEW IF EXISTS labevents_72h;

CREATE VIEW labevents_72h AS
SELECT l.*, s.stay_id
FROM labevents l
JOIN icustays s
  ON l.subject_id = s.subject_id
 AND l.hadm_id = s.hadm_id
WHERE datetime(l.charttime) >= datetime(s.intime)
  AND datetime(l.charttime) <= datetime(s.intime, '+72 hours');
""")

In [109]:
conn.executescript("""
DROP VIEW IF EXISTS outputevents_72h;

CREATE VIEW outputevents_72h AS
SELECT o.*
FROM outputevents o
JOIN icustays s USING (stay_id)
WHERE datetime(o.charttime) >= datetime(s.intime)
  AND datetime(o.charttime) <= datetime(s.intime, '+72 hours');
""")

In [110]:
hr_ids = chartevent_ids["hr"]
rr_ids = chartevent_ids["rr"]
map_ids = chartevent_ids["map"]
sbp_ids = chartevent_ids["sbp"]
dbp_ids = chartevent_ids["dbp"]
fio2_ids = chartevent_ids["fio2"]
pao2_ids = chartevent_ids["pao2"]
temp_ids = chartevent_ids["temp"]
spo2_ids = chartevent_ids["spo2"]  # if present

In [111]:
hr = ",".join(map(str, hr_ids))
rr = ",".join(map(str, rr_ids))
map_ = ",".join(map(str, map_ids))
sbp = ",".join(map(str, sbp_ids))
dbp = ",".join(map(str, dbp_ids))
temp = ",".join(map(str, temp_ids))
fio2 = ",".join(map(str, fio2_ids))
pao2 = ",".join(map(str, pao2_ids))
spo2 = ",".join(map(str, spo2_ids))

In [112]:
conn.execute("BEGIN;")

conn.executescript(f"""
DROP TABLE IF EXISTS vital_features_72h;

CREATE TABLE vital_features_72h AS
SELECT
    stay_id,

    MAX(CASE WHEN itemid IN ({hr}) THEN valuenum END) AS max_hr,
    MIN(CASE WHEN itemid IN ({hr}) THEN valuenum END) AS min_hr,
    AVG(CASE WHEN itemid IN ({hr}) THEN valuenum END) AS mean_hr,

    MAX(CASE WHEN itemid IN ({map_}) THEN valuenum END) AS max_map,
    MIN(CASE WHEN itemid IN ({map_}) THEN valuenum END) AS min_map,
    AVG(CASE WHEN itemid IN ({map_}) THEN valuenum END) AS mean_map,

    MAX(CASE WHEN itemid IN ({rr}) THEN valuenum END) AS max_rr,
    MIN(CASE WHEN itemid IN ({rr}) THEN valuenum END) AS min_rr,
    AVG(CASE WHEN itemid IN ({rr}) THEN valuenum END) AS mean_rr,

    MAX(CASE WHEN itemid IN ({spo2}) THEN valuenum END) AS max_spo2,
    MIN(CASE WHEN itemid IN ({spo2}) THEN valuenum END) AS min_spo2,
    AVG(CASE WHEN itemid IN ({spo2}) THEN valuenum END) AS mean_spo2,

    MAX(CASE WHEN itemid IN ({sbp}) THEN valuenum END) AS max_sbp,
    MIN(CASE WHEN itemid IN ({sbp}) THEN valuenum END) AS min_sbp,
    AVG(CASE WHEN itemid IN ({sbp}) THEN valuenum END) AS mean_sbp,

    MAX(CASE WHEN itemid IN ({dbp}) THEN valuenum END) AS max_dbp,
    MIN(CASE WHEN itemid IN ({dbp}) THEN valuenum END) AS min_dbp,
    AVG(CASE WHEN itemid IN ({dbp}) THEN valuenum END) AS mean_dbp,

    AVG(CASE WHEN itemid IN ({temp}) THEN 
        CASE
            WHEN valuenum > 70 THEN (valuenum - 32) * 5.0/9.0
            ELSE valuenum
        END
    END) AS mean_temp_c,

    MAX(CASE WHEN itemid IN ({fio2}) THEN
        CASE
            WHEN valuenum > 1 THEN valuenum / 100.0
            ELSE valuenum
        END
    END) AS max_fio2,

    MAX(CASE WHEN itemid IN ({pao2}) THEN valuenum END) AS max_pao2

FROM chartevents_72h
GROUP BY stay_id;
""")

conn.commit()

In [113]:
creatinine_ids = labevent_ids["creatinine"]
lactate_ids = labevent_ids["lactate"]
bilirubin_ids = labevent_ids["bilirubin"]
wbc_ids = labevent_ids["wbc"]
platelet_ids = labevent_ids["platelet"]
hemog_ids = labevent_ids["hemoglobin"]
sodium_ids = labevent_ids["sodium"]
potassium_ids = labevent_ids["potassium"]
bun_ids = labevent_ids["bun"]

In [114]:
creatinine = ",".join(map(str, creatinine_ids))
lactate = ",".join(map(str, lactate_ids))
bilirubin = ",".join(map(str, bilirubin_ids))
wbc = ",".join(map(str, wbc_ids))
platelet = ",".join(map(str, platelet_ids))
hemog = ",".join(map(str, hemog_ids))
sodium = ",".join(map(str, sodium_ids))
potassium = ",".join(map(str, potassium_ids))
bun = ",".join(map(str, bun_ids))

In [115]:
bun

'51006,51842,52647'

In [116]:
conn.executescript(f"""
DROP TABLE IF EXISTS lab_features_72h;

CREATE TABLE lab_features_72h AS
SELECT
    stay_id,

    MAX(CASE WHEN itemid IN ({creatinine}) THEN valuenum END) AS max_creatinine,
    MIN(CASE WHEN itemid IN ({creatinine}) THEN valuenum END) AS min_creatinine,
    AVG(CASE WHEN itemid IN ({creatinine}) THEN valuenum END) AS mean_creatinine,

    MAX(CASE WHEN itemid IN ({lactate}) THEN valuenum END) AS max_lactate,
    MIN(CASE WHEN itemid IN ({lactate}) THEN valuenum END) AS min_lactate,
    AVG(CASE WHEN itemid IN ({lactate}) THEN valuenum END) AS mean_lactate,

    MAX(CASE WHEN itemid IN ({bilirubin}) THEN valuenum END) AS max_bilirubin,
    MIN(CASE WHEN itemid IN ({bilirubin}) THEN valuenum END) AS min_bilirubin,
    AVG(CASE WHEN itemid IN ({bilirubin}) THEN valuenum END) AS mean_bilirubin,

    MAX(CASE WHEN itemid IN ({wbc}) THEN valuenum END) AS max_wbc,
    MIN(CASE WHEN itemid IN ({wbc}) THEN valuenum END) AS min_wbc,
    AVG(CASE WHEN itemid IN ({wbc}) THEN valuenum END) AS mean_wbc,

    MAX(CASE WHEN itemid IN ({platelet}) THEN valuenum END) AS max_platelets,
    MIN(CASE WHEN itemid IN ({platelet}) THEN valuenum END) AS min_platelets,
    AVG(CASE WHEN itemid IN ({platelet}) THEN valuenum END) AS mean_platelets,

    MAX(CASE WHEN itemid IN ({hemog}) THEN valuenum END) AS max_hemog,
    MIN(CASE WHEN itemid IN ({hemog}) THEN valuenum END) AS min_hemog,
    AVG(CASE WHEN itemid IN ({hemog}) THEN valuenum END) AS mean_hemog,

    MAX(CASE WHEN itemid IN ({sodium}) THEN valuenum END) AS max_sodium,
    MIN(CASE WHEN itemid IN ({sodium}) THEN valuenum END) AS min_sodium,
    AVG(CASE WHEN itemid IN ({sodium}) THEN valuenum END) AS mean_sodium,

    MAX(CASE WHEN itemid IN ({potassium}) THEN valuenum END) AS max_potassium,
    MIN(CASE WHEN itemid IN ({potassium}) THEN valuenum END) AS min_potassium,
    AVG(CASE WHEN itemid IN ({potassium}) THEN valuenum END) AS mean_potassium,

    MAX(CASE WHEN itemid IN ({bun}) THEN valuenum END) AS max_bun,
    MIN(CASE WHEN itemid IN ({bun}) THEN valuenum END) AS min_bun,
    AVG(CASE WHEN itemid IN ({bun}) THEN valuenum END) AS mean_bun

FROM labevents_72h
GROUP BY stay_id;
""")

In [117]:
conn.executescript("""
DROP TABLE IF EXISTS urine_features_72h;

CREATE TABLE urine_features_72h AS
SELECT
    stay_id,
    SUM(value) AS total_urine_72h
FROM outputevents_72h
GROUP BY stay_id;
""")

In [118]:
conn.executescript("""
DROP TABLE IF EXISTS demo_features;

CREATE TABLE demo_features AS
SELECT
    s.stay_id,
    p.anchor_age AS age,
    p.gender,
    a.race,
    a.admission_location,
    s.first_careunit,
    s.los,
    a.hospital_expire_flag

FROM icustays s
JOIN patients p USING (subject_id)
JOIN admissions a USING (hadm_id);
""")

In [119]:
conn.executescript("""
DROP TABLE IF EXISTS icu_features;

CREATE TABLE icu_features AS
SELECT
    d.stay_id,
    d.age,
    d.gender,
    d.race,
    d.admission_location,
    d.first_careunit,
    d.hospital_expire_flag,
    d.los,

    v.max_hr,
    v.min_hr,
    v.mean_hr,
    v.max_sbp,
    v.min_sbp,
    v.mean_sbp,
    v.max_dbp,
    v.min_dbp,
    v.mean_dbp,
    v.min_map,
    v.max_map,
    v.mean_map,
    v.max_rr,
    v.min_rr,
    v.mean_rr,
    v.min_spo2,
    v.max_spo2,
    v.mean_spo2,
    v.max_fio2,
    v.max_pao2,  
    v.mean_temp_c,

                   
    l.max_creatinine,
    l.min_creatinine,
    l.mean_creatinine,
    l.max_lactate,
    l.min_lactate,
    l.mean_lactate,
    l.max_bilirubin,
    l.min_bilirubin,
    l.mean_bilirubin,
    l.max_wbc,
    l.min_wbc,
    l.mean_wbc,
    l.max_hemog,
    l.min_hemog,
    l.mean_hemog,
    l.max_sodium,
    l.min_sodium,
    l.mean_sodium,
    l.max_potassium,
    l.min_potassium,
    l.mean_potassium,
    l.max_bun,
    l.min_bun,  
    l.mean_bun,
    l.max_platelets,        
    l.min_platelets,
    l.mean_platelets,

    u.total_urine_72h

FROM demo_features d
LEFT JOIN vital_features_72h v USING (stay_id)
LEFT JOIN lab_features_72h l USING (stay_id)
LEFT JOIN urine_features_72h u USING (stay_id);
""")


In [120]:
icu_df = pd.read_sql("SELECT * FROM icu_features;", conn)
icu_df.rename(columns={"hospital_expire_flag": "mortality_icu"}, inplace=True)
icu_df.head()

,stay_id,age,gender,race,admission_location,first_careunit,mortality_icu,los,max_hr,min_hr,...,max_potassium,min_potassium,mean_potassium,max_bun,min_bun,mean_bun,max_platelets,min_platelets,mean_platelets,total_urine_72h
0,39765666,73,F,BLACK/AFRICAN AMERICAN,EMERGENCY ROOM,Medical Intensive Care Unit (MICU),0,0.497535,80.0,68.0,...,4.5,4.2,4.300000,46.0,38.0,41.500000,223.0,188.0,212.500000,4300.0
1,37067082,55,F,WHITE,EMERGENCY ROOM,Surgical Intensive Care Unit (SICU),0,1.118032,106.0,78.0,...,3.9,3.6,3.733333,9.0,7.0,8.000000,362.0,285.0,312.666667,2745.0
2,34592300,55,F,WHITE,PHYSICIAN REFERRAL,Surgical Intensive Care Unit (SICU),0,0.948113,96.0,66.0,...,4.3,4.2,4.250000,11.0,10.0,10.500000,299.0,289.0,294.000000,2475.0
3,39698942,73,M,WHITE,TRANSFER FROM HOSPITAL,Medical/Surgical Intensive Care Unit (MICU/SICU),1,0.825266,155.0,90.0,...,5.4,3.7,4.550000,33.0,33.0,33.000000,609.0,609.0,609.000000,215.0
4,32358465,80,F,WHITE,EMERGENCY ROOM,Medical Intensive Care Unit (MICU),1,0.858576,142.0,75.0,...,5.8,5.4,5.633333,48.0,47.0,47.666667,268.0,268.0,268.000000,310.0


In [121]:
icustay_df = pd.read_sql("SELECT * FROM icustays;", conn)
print(f"Total ICU stays in sample: {len(icustay_df)}")
print(f"ICU stays with features: {len(icu_df)}")

Total ICU stays in sample: 47291
ICU stays with features: 47291


In [122]:
print("Shape of dataset:", icu_df.shape)
icu_df.info()

Shape of dataset: (47291, 57)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47291 entries, 0 to 47290
Data columns (total 57 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   stay_id             47291 non-null  int64  
 1   age                 47291 non-null  int64  
 2   gender              47291 non-null  object 
 3   race                47291 non-null  object 
 4   admission_location  47291 non-null  object 
 5   first_careunit      47291 non-null  object 
 6   mortality_icu       47291 non-null  int64  
 7   los                 47283 non-null  float64
 8   max_hr              47255 non-null  float64
 9   min_hr              47255 non-null  float64
 10  mean_hr             47255 non-null  float64
 11  max_sbp             47115 non-null  float64
 12  min_sbp             47115 non-null  float64
 13  mean_sbp            47115 non-null  float64
 14  max_dbp             47112 non-null  float64
 15  min_dbp             471